<div dir="rtl" style="text-align:right">
<h1>چند Head، یک خروجی</h1><p style="text-align:right"><b>پرسش آزمایش:</b> چگونه چند مسیر توجه دوباره به یک نمایش Cتایی برمی‌گردند؟</p><p style="text-align:right">پیش‌نیاز: <a href="http://127.0.0.1:8000/part-06/chapter-01/35-split-heads.html"><bdi dir="ltr">35-split-heads</bdi></a>، <a href="http://127.0.0.1:8000/part-06/chapter-01/36-merge-heads.html"><bdi dir="ltr">36-merge-heads</bdi></a></p><p style="text-align:right">این دفتر مستقل است و به اجرای دفتر دیگری نیاز ندارد. از بالا به پایین اجرا کنید؛ برای اجرای دوباره از ابتدا، Kernel را Restart و سپس Run All کنید. برای بازکردن لینک درس‌ها، سرور کتاب باید روی پورت ۸۰۰۰ اجرا شده باشد؛ راهنمای نصب در <a href="../../docs/NOTEBOOKS.md"><bdi dir="ltr">docs/NOTEBOOKS.md</bdi></a> است.</p>
</div>

In [ ]:
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "mini_gpt" / "model.py").is_file()
             and (p / "data" / "sample.txt").is_file()), None)
if ROOT is None:
    raise RuntimeError("Keep notebooks inside the extracted project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Python:", sys.executable)
print("Project:", ROOT)

import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(17)

def inspect(name, value):
    print(name, "shape =", tuple(value.shape),
          "dtype =", value.dtype, "device =", value.device)


<div dir="rtl" style="text-align:right">
<p style="text-align:right">خروجی Attention واقعی پروژه را با محاسبهٔ خودمان مقایسه می‌کنیم. مرحله‌ها را با همان وزن‌ها بازسازی می‌کنیم تا تفاوتی از وزن‌های تصادفی وارد مقایسه نشود. B تعداد نمونه، T طول، C تعداد ویژگی، H تعداد Head و D=C/H است. قبل از اجرا، شکل خروجی یک‌جای QKV و جدول وزن‌های Attention را بنویسید.</p>
</div>

In [ ]:
from mini_gpt.config import ModelConfig
from mini_gpt.attention import CausalSelfAttention
B,T,C,H = 2,5,12,3
D = C//H
config = ModelConfig(vocab_size=12,context_length=8,embedding_dim=C,
                     num_heads=H,num_layers=1,dropout=0.)
attention = CausalSelfAttention(config).eval()
x = torch.randn(B,T,C)
trace = {}
with torch.no_grad():
    output, weights = attention(x, return_weights=True, trace=trace)
    combined_qkv = attention.qkv(x)
    q_flat,k_flat,v_flat = combined_qkv.chunk(3,dim=-1)
    for name, flat in zip(("q","k","v"),(q_flat,k_flat,v_flat)):
        heads = flat.reshape(B,T,H,D).transpose(1,2)
        torch.testing.assert_close(heads,trace[name])
    per_head = weights @ trace["v"]
    torch.testing.assert_close(per_head,trace["weighted_values"])
    merged = per_head.transpose(1,2).contiguous().view(B,T,C)
    projected = attention.output(merged)
    torch.testing.assert_close(projected,output)
for name, value in [("X",x),("QKV",combined_qkv),("Q before split",q_flat),
                    ("Q heads",trace["q"]),("weights",weights),
                    ("weighted V",per_head),("merged",merged),("projected",output)]:
    inspect(name,value)


In [ ]:
fig, axes = plt.subplots(1,H,figsize=(3*H,3),squeeze=False)
for head, ax in enumerate(axes[0]):
    ax.imshow(weights[0,head],vmin=0,vmax=1,cmap="Blues")
    ax.set(title=f"Head {head}",xlabel="Key",ylabel="Query")
plt.tight_layout()
plt.show()
torch.testing.assert_close(weights.sum(-1),torch.ones(B,H,T))
assert torch.count_nonzero(weights.triu(1)) == 0
try:
    ModelConfig(vocab_size=12,embedding_dim=10,num_heads=3)
except ValueError as error:
    print("Expected invalid C/H:",error)
else:
    raise AssertionError("C must be divisible by H")


<div dir="rtl" style="text-align:right">
<p style="text-align:right"><b>تمرین:</b> با C=12، H را به ۱ و سپس ۴ تغییر دهید و کل دفتر را از نو اجرا کنید. D و شکل جدول وزن چه می‌شوند؟ شمار پارامترهای QKV و Output projection را مقایسه کنید. انتظار نداریم Headهای تصادفی از پیش نقش‌های زبانی معنادار داشته باشند. برابرشدن Shape کافی نبود؛ به همین دلیل مقدارهای ادغام و Projection را هم آزمودیم.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2>برگشت به کتاب</h2><p style="text-align:right">پیش‌بینی و نتیجهٔ اجرا را کنار هم بنویسید؛ اگر تفاوتی داشتند، دلیلش را توضیح دهید. سپس به <a href="http://127.0.0.1:8000/part-06/chapter-01/36-merge-heads.html">درس مرتبط</a> برگردید و نتیجه را با توضیح آن مقایسه کنید.</p>
</div>